# 16 — AI Agent Security Foundations & Threat Modeling

## Learning requirements
Sau notebook này bạn phải:
- phân biệt AI safety, AI security và traditional AppSec;
- xác định asset, actor, trust boundary, entry point và blast radius;
- dùng OWASP Top 10 for Agentic Applications 2026 làm threat checklist;
- hiểu NIST AI RMF theo bốn function: Govern, Map, Measure, Manage;
- biến threat thành control và security test, không dừng ở checklist.

## Security baseline 2026
OWASP Agentic Top 10:
1. ASI01 — Agent Goal Hijack
2. ASI02 — Tool Misuse & Exploitation
3. ASI03 — Identity & Privilege Abuse
4. ASI04 — Agentic Supply Chain Vulnerabilities
5. ASI05 — Unexpected Code Execution
6. ASI06 — Memory & Context Poisoning
7. ASI07 — Insecure Inter-Agent Communication
8. ASI08 — Cascading Failures
9. ASI09 — Human-Agent Trust Exploitation
10. ASI10 — Rogue Agents

> Frameworks là taxonomy để đặt câu hỏi đúng; security control cuối cùng phải phù hợp architecture thực tế của hệ thống.

## Threat-model mental model

```text
User / External Content
          |
          v
      Agent Runtime
       /   |    \
      /    |     \
   Memory  RAG    Tools ----> External systems
     |      |       |
     +------+-------+
            |
      Trust boundaries
```

Đối với mỗi boundary, hỏi:
- Ai kiểm soát input?
- Dữ liệu có trusted không?
- Principal nào đang thực thi?
- Credential nào được dùng?
- Action có reversible không?
- Nếu component bị compromise, blast radius là gì?

In [ ]:
from dataclasses import dataclass
from enum import IntEnum

class Level(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4

@dataclass(frozen=True)
class Risk:
    id: str
    asset: str
    threat: str
    likelihood: Level
    impact: Level
    control: str
    security_test: str

    @property
    def score(self) -> int:
        return int(self.likelihood) * int(self.impact)

risks = [
    Risk("R-001", "project source code", "untrusted repository text redirects agent goal", Level.HIGH, Level.HIGH, "treat repository text as untrusted data and gate privileged tools", "malicious README must not change tool authorization"),
    Risk("R-002", "user memory", "cross-tenant memory read", Level.MEDIUM, Level.CRITICAL, "tenant+user namespace authorization", "user A cannot read user B namespace"),
]

for risk in sorted(risks, key=lambda r: r.score, reverse=True):
    print(risk.id, risk.score, risk.threat)

## NIST AI RMF mapping exercise

Với mỗi critical risk, ghi rõ:
- **Govern**: owner/policy/accountability nào chịu trách nhiệm?
- **Map**: system context, actor, impact và dependency nào liên quan?
- **Measure**: metric/test nào chứng minh risk đang được kiểm soát?
- **Manage**: control, monitoring, fallback và response nào được áp dụng?

Không cần biến course thành compliance course; mục tiêu là xây thói quen risk management xuyên suốt lifecycle.

## Exercise — Threat model Interview Agent

Threat-model tối thiểu các component:
- source-code scanner;
- vector store / RAG;
- interview thread state;
- long-term memory;
- Git/MCP tools;
- subagents;
- human approval UI;
- LangSmith traces;
- deployment credentials.

Mỗi risk phải map tới ít nhất một OWASP ASI category, một preventive/detective control và một test case.

## Required output
Tạo:
- `artifacts/security/threat-model.md`
- `artifacts/security/risk-register.csv`
- architecture diagram có trust boundaries.

## Done criteria
- Có coverage ASI01–ASI10.
- Không có risk chỉ ghi 'use guardrail' mà không define guardrail ở đâu.
- Critical/high risks đều có owner + control + measurable test.
- Bạn giải thích được blast radius nếu agent, memory hoặc tool credential bị compromise.